# travel 路线规划算法实验室

GA-TSP 参数寻优与新旧算法对比。数据：`attractions.csv`（init_complete.sql 提取的真实 20 景点）。

In [ ]:
import math, random, itertools, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['Microsoft YaHei']
plt.rcParams['axes.unicode_minus']=False

import io, os, base64, urllib.request, urllib.parse, getpass

# 数据流转链路：GitHub master（事实源）→ Actions 自动镜像 → Gitee → notebook
# 私有仓库 raw 不支持令牌参数，走 Gitee OpenAPI v5（base64）；令牌从环境变量 GITEE_TOKEN 读取，不进代码
REPO='travel-regulations/site-project'
DATA_PATH='experiments/travel-algo-lab/attractions.csv'
LOCAL='attractions.csv'

def load_attractions():
    if os.path.exists(LOCAL):
        print('使用本地 attractions.csv')
        return pd.read_csv(LOCAL)
    token=os.environ.get('GITEE_TOKEN') or getpass.getpass('Gitee 私人令牌: ')
    url=(f'https://gitee.com/api/v5/repos/{REPO}/contents/{DATA_PATH}'
         f'?access_token={urllib.parse.quote(token)}&ref=master')
    print('从 Gitee 镜像拉取（API v5）...')
    resp=urllib.request.urlopen(url,timeout=15)
    content=base64.b64decode(json.load(resp)['content']).decode('utf-8-sig')
    return pd.read_csv(io.StringIO(content))

df=load_attractions()
display(df.head())

def haversine(lon1,lat1,lon2,lat2,R=6371.0):
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

def tour_distance(pts):
    return sum(haversine(*pts[i],*pts[i+1]) for i in range(len(pts)-1))


## 新旧算法实现（可配置开关）

In [ ]:
class GA:
    """可配置 GA-TSP：nn_seed(贪心最近邻种子)/early_stop/2-opt 均可开关。
    旧算法 = 关闭全部三项 + 固定 200 代。"""
    def __init__(self, pop=100, gen=200, cx=0.8, mut=0.02, elite=5,
                 nn_seed=True, early_stop=30, two_opt=True, seed=42):
        self.pop,self.gen,self.cx,self.mut,self.elite=pop,gen,cx,mut,elite
        self.nn_seed,self.early_stop,self.two_opt,self.seed=nn_seed,early_stop,two_opt,seed

    def solve(self, pts):
        n=len(pts)
        if n<=2: return list(range(n)), self._dist_tour(pts,list(range(n))), 0
        rng=random.Random(self.seed)
        dist=[[haversine(*pts[i],*pts[j]) for j in range(n)] for i in range(n)]
        def tour_d(t): return sum(dist[t[i]][t[i+1]] for i in range(n-1))
        def fit(t): return 1/(1+tour_d(t))

        # 初始化：0号=贪心最近邻，其余随机
        popu=[[rng.sample(range(n),n)][0] for _ in range(self.pop)]
        if self.nn_seed:
            cur=rng.randrange(n); vis={cur}; tour=[cur]
            for _ in range(n-1):
                nxt=min((j for j in range(n) if j not in vis),key=lambda j:dist[cur][j])
                tour.append(nxt); vis.add(nxt); cur=nxt
            popu[0]=tour

        best,best_f,stag,gens,gen_best_pre=None,-1,0,0,-1
        for g in range(self.gen):
            gens=g+1
            fits=[fit(t) for t in popu]
            gen_best=max(fits)   # 先记当代最优，再更新历史（否则早停计数恒真）
            for t,f in zip(popu,fits):
                if f>best_f: best_f,best=f,t[:]
            if self.early_stop and g>=self.gen//2:
                stag=0 if best_f>gen_best_pre else stag+1
                if stag>=self.early_stop: break
            gen_best_pre=gen_best
            order=sorted(range(self.pop),key=lambda i:-fits[i])
            new=[popu[i][:] for i in order[:self.elite]]
            def tournament():
                cand=[rng.randrange(self.pop) for _ in range(5)]
                return popu[max(cand,key=lambda i:fits[i])][:]
            def ox(p1,p2):
                a,b=sorted(rng.sample(range(n),2)); child=[-1]*n
                child[a:b+1]=p1[a:b+1]
                pos=(b+1)%n
                for i in range(n):
                    g2=p2[(b+1+i)%n]
                    if g2 not in child:
                        child[pos]=g2; pos=(pos+1)%n
                return child
            while len(new)<self.pop:
                p1,p2=tournament(),tournament()
                child=ox(p1,p2) if rng.random()<self.cx else p1[:]
                if rng.random()<self.mut:
                    i,j=rng.sample(range(n),2); child[i],child[j]=child[j],child[i]
                new.append(child)
            popu=new

        if self.two_opt:
            # 开放路径 2-opt：不含闭环边（与 Java 版一致），否则会优化错误的度量
            improved=True
            while improved:
                improved=False
                for i in range(n-2):
                    for j in range(i+2,n-1):
                        d_before=dist[best[i]][best[i+1]]+dist[best[j]][best[j+1]]
                        d_after=dist[best[i]][best[j]]+dist[best[i+1]][best[j+1]]
                        if d_after<d_before-1e-9:
                            best[i+1:j+1]=best[i+1:j+1][::-1]; improved=True
        return best,tour_d(best),gens

    def _dist_tour(self,pts,t):
        return sum(haversine(*pts[i],*pts[i+1]) for i in range(len(t)-1))


## 对比实验：旧 GA vs 新 GA（NN种子+早停+2-opt）

In [ ]:
# 旧算法 vs 新算法（真实 20 景点全集）
pts=list(zip(df.lng,df.lat))
old=GA(nn_seed=False,early_stop=0,two_opt=False,gen=200,seed=1)
new=GA(seed=1)
# 多种子平均，避免单次运气
import statistics
def avg(cfg,runs=10):
    ds=[]
    for s in range(runs):
        g=GA(**{**cfg,'seed':s})
        _,d,_=g.solve(pts)
        ds.append(d)
    return statistics.mean(ds),statistics.stdev(ds),ds

naive=tour_distance(pts)
old_m,old_s,_=avg({'nn_seed':False,'early_stop':0,'two_opt':False})
new_m,new_s,_=avg({})
print(f"朴素顺序: {naive:.1f} km")
print(f"旧GA(200代): {old_m:.1f} ± {old_s:.1f} km")
print(f"新GA(NN种子+早停+2opt): {new_m:.1f} ± {new_s:.1f} km")
print(f"新 vs 旧 提升: {(old_m-new_m)/old_m*100:.1f}%")


## 参数网格寻优

In [ ]:
# 参数网格寻优（在真实数据上）
import itertools
results=[]
for pop,gen,cx,mut in itertools.product([50,100,150],[100,200,300],[0.6,0.8],[0.01,0.02,0.05]):
    g=GA(pop=pop,gen=gen,cx=cx,mut=mut,seed=0)
    t0=time.time()
    _,d,_=g.solve(pts)
    results.append({'pop':pop,'gen':gen,'cx':cx,'mut':mut,'dist':round(d,1),'sec':round(time.time()-t0,2)})
grid=pd.DataFrame(results).sort_values('dist')
display(grid.head(10))
grid.to_csv('grid_results.csv',index=False,encoding='utf-8-sig')


## 规模扩展（合成数据 n=10/30/50）

In [ ]:
# 规模扩展实验：合成地理簇数据 n=10/30/50（每簇内点彼此靠近，模拟同城/邻城）
def synthetic(n_points, n_clusters=3, spread=0.3, seed=7):
    rng=random.Random(seed)
    centers=[(rng.uniform(100,122),rng.uniform(22,41)) for _ in range(n_clusters)]
    pts=[]
    for i in range(n_points):
        cx,cy=centers[i%n_clusters]
        pts.append((cx+rng.gauss(0,spread),cy+rng.gauss(0,spread)))
    return pts

rows=[]
for n in [10,30,50]:
    pts=synthetic(n)
    t0=time.time(); _,d_old,g_old=GA(nn_seed=False,early_stop=0,two_opt=False,gen=200,seed=0).solve(pts); t_old=time.time()-t0
    t0=time.time(); _,d_new,g_new=GA(seed=0).solve(pts); t_new=time.time()-t0
    rows.append({'n':n,'old_km':round(d_old,1),'new_km':round(d_new,1),
                 'improve_%':round((d_old-d_new)/d_old*100,1),
                 'old_gens':g_old,'new_gens':g_new,'old_sec':round(t_old,2),'new_sec':round(t_new,2)})
pd.DataFrame(rows)


## 优化路线可视化

In [ ]:
# 收敛对比可视化（真实数据）
pts=list(zip(df.lng,df.lat))
# 简化：绘 2-opt 前后路径
g=GA(seed=0)
tour,d,_=g.solve(pts)
order=[df.iloc[i]['name'] for i in tour]+[df.iloc[tour[0]]['name']]
fig,ax=plt.subplots(figsize=(10,6))
lon=[df.lng.iloc[i] for i in tour]+[df.lng.iloc[tour[0]]]
lat=[df.lat.iloc[i] for i in tour]+[df.lat.iloc[tour[0]]]
ax.plot(lon,lat,'-o',color='steelblue')
for i,t in enumerate(tour):
    ax.annotate(df.name.iloc[t],(df.lng.iloc[t],df.lat.iloc[t]),fontsize=8)
ax.set_title(f'新算法优化路线（总距离 {d:.0f} km，含闭环返回）')
plt.tight_layout(); plt.savefig('route_result.png',dpi=120)
plt.show()
